# Baseline — Seasonal Naive (lag-52)

Prediction = same store/dept sales 52 weeks earlier (fallbacks: lag 53, lag 51, series median, dept median).

This is the anchor: **any model scoring worse than this is broken, not 'still tuning'.** Also debugs the whole Kaggle submission pipeline on day one.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))  # repo root on path

import numpy as np
import pandas as pd
import mlflow

from src.data import load_raw, make_submission
from src.metrics import wmae_report
from src.validation import FOLDS, split_fold
from src.mlflow_utils import setup_mlflow

train, test, features, stores = load_raw()
print(train.shape, test.shape)

setup_mlflow("Baseline_Training")

In [ ]:
def seasonal_naive(history, target):
    hist = history[["Store", "Dept", "Date", "Weekly_Sales"]]
    out = target[["Store", "Dept", "Date"]].copy()
    for L in (52, 53, 51):
        lagged = hist.copy()
        lagged["Date"] = lagged["Date"] + pd.Timedelta(days=7 * L)
        out = out.merge(lagged.rename(columns={"Weekly_Sales": f"lag_{L}"}),
                        on=["Store", "Dept", "Date"], how="left")
    pred = out["lag_52"].fillna(out["lag_53"]).fillna(out["lag_51"])
    ser_med = hist.groupby(["Store", "Dept"])["Weekly_Sales"].median().rename("ser_med")
    dep_med = hist.groupby("Dept")["Weekly_Sales"].median().rename("dep_med")
    out = out.join(ser_med, on=["Store", "Dept"]).join(dep_med, on="Dept")
    pred = pred.fillna(out["ser_med"]).fillna(out["dep_med"]).fillna(0.0)
    return pred.to_numpy()

In [ ]:
for fold in (1, 2):
    tr, va = split_fold(train, fold)
    rep = wmae_report(va["Weekly_Sales"], seasonal_naive(tr, va), va["IsHoliday"])
    with mlflow.start_run(run_name=f"SeasonalNaive_Fold{fold}"):
        mlflow.log_params({"model": "seasonal_naive_lag52", "fold": fold, **FOLDS[fold]})
        mlflow.log_metrics(rep)
    print(f"fold {fold}: {rep}")

In [ ]:
# Kaggle submission (upload manually or: kaggle competitions submit -c walmart-recruiting-store-sales-forecasting -f ...)
sub = test.copy()
sub["pred"] = seasonal_naive(train, test)
make_submission(sub, "pred", "submission_baseline.csv")
print("wrote submission_baseline.csv")